In [1]:
import os, sys
from pyspark.sql import functions as SF
import duckdb

In [3]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [2]:
# Cria a conexão com o DuckDB

con = duckdb.connect(database=':memory:')

con.execute("SET max_memory = '10GB';")
con.execute("SET threads = 4;")

In [4]:
PROJECT_PATH    = os.getcwd()
ROOT_DATA_PATH  = "C:\\Marco Conti\\Projetos\\Dados\\"

Seleciona a grade de municípios


Faz a junção dos municípios e temperatura aplicando o peso ponderado para o município

In [10]:
path_ponto_orvalho  = r"C:\Marco Conti\Projetos\Dados\ERA5-Umidade\arquivos_ponto_orvalho_parquet\*.parquet"
duckdb.sql(f"DESCRIBE SELECT * FROM '{path_ponto_orvalho}'").show()

path_temperatura    = r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_parquet_2015_2026\*.parquet"
duckdb.sql(f"DESCRIBE SELECT * FROM '{path_temperatura}'").show()

path_municipios     = r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet"
duckdb.sql(f"DESCRIBE SELECT * FROM '{path_municipios}'").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ data_medicao   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ longitude      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ expver         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ indicador      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ valor          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ unidade_medida │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key  

In [ ]:
# path_ponto_orvalho  = r"C:\Marco Conti\Projetos\Dados\ERA5-Umidade\arquivos_ponto_orvalho_parquet\*.parquet"
# path_temperatura    = r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_parquet_2015_2026\*.parquet"
# path_municipios     = r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet"

path_write_parquet  = r"C:\Marco Conti\Projetos\Dados\umidade_por_municipios_SEM_georrefenciamento.parquet"


duckdb_sql = \
   f"""COPY (WITH medias_municipio AS (
                -- 1. Junta Temperatura e Ponto de Orvalho por lat/lon/data e calcula as médias ponderadas por município
                SELECT 
                    m.code_muni,
                    m.name_muni,
                    m.abbrev_state AS uf,
                    t.data_medicao,
                    SUM(t.valor * m.fator_peso_municipio) AS t_celsius,
                    SUM(d.valor * m.fator_peso_municipio) AS td_celsius
                FROM '{path_temperatura}' t
                JOIN '{path_ponto_orvalho}' d 
                    ON  t.latitude = d.latitude 
                    AND t.longitude = d.longitude 
                    AND t.data_medicao = d.data_medicao
                JOIN '{path_municipios}' m 
                    ON  t.latitude = m.latitude_centro 
                    AND t.longitude = m.longitude_centro
                GROUP BY 
                    m.code_muni, 
                    m.name_muni, 
                    m.abbrev_state, 
                    t.data_medicao
            )
            -- 2. Aplica Magnus-Tetens nos valores municipais agregados e aplica a trava de 100%
            SELECT 
                code_muni,
                name_muni,
                uf,
                data_medicao
                ROUND(t_celsius, 2) AS temperatura_ar,
                ROUND(td_celsius, 2) AS ponto_orvalho,
                LEAST(
                    ROUND(
                        100.0 * EXP( (17.67 * td_celsius)/(243.5 + td_celsius) - (17.67 * t_celsius)/(243.5 + t_celsius) ), 
                        2
                    ), 
                    100.0
                ) AS umidade_ar
            FROM medias_municipio
            )
         TO '{path_write_parquet}' 
         (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
    """
# (FORMAT CSV, HEADER TRUE, PARTITION_BY (ANO), OVERWRITE_OR_IGNORE);

print("Executando transformação e gravação em PARQUET via DuckDB...")
con.execute(duckdb_sql)
print("Concluído!")


Executando transformação e gravação em PARQUET via DuckDB...
Concluído!


In [34]:
import duckdb

arquivo = path_write_parquet

# # Exibe o Schema
# print("--- SCHEMA ---")
# duckdb.sql(f"DESCRIBE SELECT * FROM '{arquivo}'").show()

# # Exibe Amostra de 5 linhas
# print("\n--- AMOSTRA ---")
# duckdb.sql(f"SELECT * FROM '{arquivo}' where data_medicao = '2025-09-01' and uf = 'SP' LIMIT 20").show()


path_temp_media = r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_media_ano_mes.parquet"
duckdb.sql(f"SELECT * FROM '{path_temp_media}'").show()

┌───────┬───────┬───────────┬────────────────────────┬────────┬────────────────┐
│  ano  │  mes  │ code_muni │       indicador        │ valor  │ unidade_medida │
│ int32 │ int32 │  double   │        varchar         │ double │    varchar     │
├───────┼───────┼───────────┼────────────────────────┼────────┼────────────────┤
│  2015 │    12 │ 5007695.0 │ Temperatura média (°C) │  23.89 │ Celsius        │
│  2018 │     5 │ 2103703.0 │ Temperatura média (°C) │  25.64 │ Celsius        │
│  2018 │     9 │ 3116159.0 │ Temperatura média (°C) │  22.25 │ Celsius        │
│  2015 │     1 │ 2901106.0 │ Temperatura média (°C) │  23.97 │ Celsius        │
│  2015 │     3 │ 3151404.0 │ Temperatura média (°C) │  21.88 │ Celsius        │
│  2015 │     3 │ 3100807.0 │ Temperatura média (°C) │  20.74 │ Celsius        │
│  2015 │     3 │ 4315800.0 │ Temperatura média (°C) │  18.86 │ Celsius        │
│  2015 │     3 │ 3146503.0 │ Temperatura média (°C) │  21.23 │ Celsius        │
│  2015 │     3 │ 2400505.0 

In [37]:
path_umidade_FULL = path_write_parquet
path_umidade_write = r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_umidade\umidade_media_ano_mes.parquet"

query_umidade_ano_mes = \
   f"""Copy(Select YEAR(cast(data_medicao as date)) as ano
                  ,MONTH(cast(data_medicao as date)) as mes
                  ,code_muni
                  ,'Umidade média (%)' indicador
                  ,round(avg(umidade_ar),2) as valor
                  ,'%' as unidade_medida
              from '{path_umidade_FULL}'
              group by YEAR(cast(data_medicao as date))
                      ,MONTH(cast(data_medicao as date))
                      ,code_muni)
         TO '{path_umidade_write}' 
         (FORMAT PARQUET, OVERWRITE_OR_IGNORE);

    """

duckdb.sql(query_umidade_ano_mes)



In [39]:
duckdb.sql(f"Select * from '{path_umidade_write}'")

┌───────┬───────┬───────────┬───────────────────┬────────┬────────────────┐
│  ano  │  mes  │ code_muni │     indicador     │ valor  │ unidade_medida │
│ int64 │ int64 │  double   │      varchar      │ double │    varchar     │
├───────┼───────┼───────────┼───────────────────┼────────┼────────────────┤
│  2015 │     1 │ 3138807.0 │ Umidade média (%) │  71.59 │ %              │
│  2015 │     3 │ 2933257.0 │ Umidade média (%) │  94.74 │ %              │
│  2015 │     3 │ 4301073.0 │ Umidade média (%) │  90.05 │ %              │
│  2015 │     3 │ 3138609.0 │ Umidade média (%) │  89.94 │ %              │
│  2015 │     5 │ 1400704.0 │ Umidade média (%) │  90.31 │ %              │
│  2015 │     5 │ 1711506.0 │ Umidade média (%) │  85.53 │ %              │
│  2015 │     8 │ 4214102.0 │ Umidade média (%) │  90.56 │ %              │
│  2015 │     8 │ 3117405.0 │ Umidade média (%) │  88.15 │ %              │
│  2015 │     8 │ 3505708.0 │ Umidade média (%) │  72.71 │ %              │
│  2015 │   